# Model Packaging: joblib, pickle, and ONNX

Trained models must be serialised for deployment. This notebook covers:
1. **joblib / pickle** -- Python-native serialisation
2. **ONNX** -- open, framework-agnostic format
3. **Best practices** for model packaging

In [ ]:
import numpy as np
import os, tempfile, time
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib
import pickle

try:
    import onnxruntime as ort
    from skl2onnx import to_onnx
    HAS_ONNX = True
except ImportError:
    HAS_ONNX = False
    print('ONNX tools not installed -- pip install onnxruntime skl2onnx')

In [ ]:
# Train a model
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42
)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
baseline_acc = accuracy_score(y_test, rf.predict(X_test))
print(f"Baseline accuracy: {baseline_acc:.4f}")

## 1. joblib and pickle

**joblib** is optimised for numpy arrays and is the recommended way to serialise scikit-learn models.

In [ ]:
tmpdir = tempfile.mkdtemp()

# joblib
joblib_path = os.path.join(tmpdir, 'model.joblib')
joblib.dump(rf, joblib_path)
rf_loaded = joblib.load(joblib_path)
acc_joblib = accuracy_score(y_test, rf_loaded.predict(X_test))
size_joblib = os.path.getsize(joblib_path) / 1024

# pickle
pickle_path = os.path.join(tmpdir, 'model.pkl')
with open(pickle_path, 'wb') as f:
    pickle.dump(rf, f)
with open(pickle_path, 'rb') as f:
    rf_pkl = pickle.load(f)
acc_pickle = accuracy_score(y_test, rf_pkl.predict(X_test))
size_pickle = os.path.getsize(pickle_path) / 1024

print(f"{'Format':<10} {'Size (KB)':>10} {'Accuracy':>10}")
print(f"{'joblib':<10} {size_joblib:>10.1f} {acc_joblib:>10.4f}")
print(f"{'pickle':<10} {size_pickle:>10.1f} {acc_pickle:>10.4f}")

## 2. ONNX (Open Neural Network Exchange)

ONNX provides a **framework-agnostic** format. Benefits:
- Run models trained in Python with C++, Java, JavaScript, etc.
- Hardware-accelerated inference via ONNX Runtime
- Smaller file sizes and faster inference

In [ ]:
if HAS_ONNX:
    # Convert sklearn model to ONNX
    onnx_model = to_onnx(rf, X_train[:1].astype(np.float32))
    onnx_path = os.path.join(tmpdir, 'model.onnx')
    with open(onnx_path, 'wb') as f:
        f.write(onnx_model.SerializeToString())
    
    size_onnx = os.path.getsize(onnx_path) / 1024
    
    # Inference with ONNX Runtime
    sess = ort.InferenceSession(onnx_path)
    input_name = sess.get_inputs()[0].name
    onnx_pred = sess.run(None, {input_name: X_test.astype(np.float32)})[0]
    acc_onnx = accuracy_score(y_test, onnx_pred)
    
    print(f"{'ONNX':<10} {size_onnx:>10.1f} KB  accuracy={acc_onnx:.4f}")
else:
    print('Install onnxruntime and skl2onnx to try ONNX export.')

In [ ]:
# Inference speed comparison
if HAS_ONNX:
    n_iters = 1000
    
    # sklearn
    start = time.time()
    for _ in range(n_iters):
        rf.predict(X_test)
    sklearn_time = (time.time() - start) / n_iters * 1000
    
    # ONNX
    start = time.time()
    X_test_f32 = X_test.astype(np.float32)
    for _ in range(n_iters):
        sess.run(None, {input_name: X_test_f32})
    onnx_time = (time.time() - start) / n_iters * 1000
    
    print(f"Sklearn inference: {sklearn_time:.3f} ms/batch")
    print(f"ONNX inference:    {onnx_time:.3f} ms/batch")
    print(f"Speedup: {sklearn_time/onnx_time:.1f}x")

## 3. Best Practices

| Practice | Why |
|----------|-----|
| Version your models | Track which model is in production |
| Save preprocessing too | The pipeline must match at inference time |
| Pin library versions | Pickle/joblib are Python-version-sensitive |
| Use ONNX for cross-platform | Decouples training from serving |
| Test loaded models | Verify accuracy after serialisation |

In [ ]:
# Save a complete pipeline (preprocessing + model)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pipeline = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=100, random_state=42))
pipeline.fit(X_train, y_train)

pipe_path = os.path.join(tmpdir, 'pipeline.joblib')
joblib.dump(pipeline, pipe_path)

pipe_loaded = joblib.load(pipe_path)
acc_pipe = accuracy_score(y_test, pipe_loaded.predict(X_test))
print(f"Pipeline accuracy after reload: {acc_pipe:.4f}")
print(f"Pipeline size: {os.path.getsize(pipe_path)/1024:.1f} KB")

## Key Takeaways

- **joblib** is the standard for scikit-learn model serialisation.
- **ONNX** enables cross-framework, cross-language deployment with faster inference.
- Always save the **complete pipeline** (preprocessing + model) together.
- Validate model accuracy after deserialisation.

**Next:** API deployment with FastAPI.